# Diabetes status classification — Decision tree

**Dataset:** `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` from DTU Learn. Set `DATA_PATH` below to its local location. Run cells in order. Install dependencies if needed: `pip install numpy pandas scikit-learn matplotlib seaborn`.

The label `Diabetes_binary` denotes reported (pre)diabetes **at survey time**. The dataset was deliberately balanced 50/50. A predicted probability is therefore a model score for this sampled population, **not** an individual's future absolute risk. Feature contributions describe the model, not causal effects or medical advice. Keep the held-out test set untouched while choosing settings.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, average_precision_score, RocCurveDisplay,
                             brier_score_loss)
from sklearn.inspection import permutation_importance


In [ ]:
DATA_PATH = Path('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
SEED = 42
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Set DATA_PATH to the CSV downloaded from DTU Learn: {DATA_PATH.resolve()}')
df = pd.read_csv(DATA_PATH)
target = 'Diabetes_binary'
if target not in df:
    raise KeyError(f'Missing {target}; columns: {df.columns.tolist()}')
if df.columns.duplicated().any():
    raise ValueError('Duplicate column names')
X = df.drop(columns=target).apply(pd.to_numeric, errors='raise')
y = df[target].astype(int)
assert set(y.unique()) == {0, 1}, 'Expected binary label 0/1'
print('Rows, features:', X.shape, '| positive fraction:', round(y.mean(), 3))
print('Missing values:', int(X.isna().sum().sum()))
display(X.head())

# Identical split in both notebooks. Split before fitting any imputation/scaling.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
features = X.columns.tolist()


## Train a shallow, readable decision tree

Imputation is fitted only on training data. `max_depth` and `min_samples_leaf` constrain complexity; these are baseline settings, not tuned on the test set.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree

model = make_pipeline(SimpleImputer(strategy='median'),
                      DecisionTreeClassifier(max_depth=4, min_samples_leaf=100,
                                             random_state=SEED))
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]
plt.figure(figsize=(20, 9))
plot_tree(model[-1], feature_names=features, class_names=['No', 'Yes'],
          filled=True, rounded=True, fontsize=8, max_depth=3)
plt.title('Decision tree (top four levels)'); plt.show()


## Held-out evaluation

In [ ]:
def evaluate(y_true, probability, threshold=0.5):
    prediction = (probability >= threshold).astype(int)
    print(classification_report(y_true, prediction, target_names=['no (pre)diabetes', '(pre)diabetes']))
    print(f'ROC AUC: {roc_auc_score(y_true, probability):.3f}')
    print(f'Average precision: {average_precision_score(y_true, probability):.3f}')
    print(f'Brier score (balanced sample only): {brier_score_loss(y_true, probability):.3f}')
    ConfusionMatrixDisplay.from_predictions(y_true, prediction, display_labels=['No', 'Yes'], cmap='Blues')
    plt.title('Held-out test set, threshold 0.5'); plt.show()
    RocCurveDisplay.from_predictions(y_true, probability); plt.show()

evaluate(y_test, probabilities)


## Global explanation: permutation importance
### We check for collinearity between features.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform

# 1. Compute Pairwise Absolute Correlation Matrix across ALL features
corr_matrix = X_test.corr(method='pearson').abs()

# 2. Convert Correlation to Distance Matrix (Distance = 1 - |r|)
# Fill diagonal with 0 to prevent tiny floating-point rounding errors
dist_matrix = 1.0 - corr_matrix
np.fill_diagonal(dist_matrix.values, 0)

# Convert to condensed distance vector required by SciPy
condensed_dist = squareform(dist_matrix)

# 3. Perform Hierarchical Agglomerative Clustering (Complete Linkage)
linkage_matrix = linkage(condensed_dist, method='complete')

# 4. Plot the Dendrogram
plt.figure(figsize=(13, 6))
dendrogram(
    linkage_matrix,
    labels=X_test.columns,
    leaf_rotation=90,
    leaf_font_size=11,
    color_threshold=0.65,  # Highlights clusters with |r| >= 0.35
)

# Draw cutoff line corresponding to correlation threshold |r| = 0.35
plt.axhline(
    y=0.65, color='red', linestyle='--', label='Cutoff Threshold (|r| ≥ 0.35)'
)

plt.title(
    'Hierarchical Clustering Dendrogram Across All 21 Features',
    fontsize=14,
    fontweight='bold',
)
plt.ylabel('Distance (1 - |Correlation|)')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# 5. Automatically Extract and Print Clusters
distance_threshold = 0.65  # Equivalent to |r| >= 0.35
cluster_ids = fcluster(
    linkage_matrix, t=distance_threshold, criterion='distance'
)

cluster_groups = {}
for feature, cid in zip(X_test.columns, cluster_ids):
  cluster_groups.setdefault(cid, []).append(feature)

print(
    '=== Mathematically Derived Feature Clusters (|r| ≥ 0.35) ===\n'
)
for cid, members in cluster_groups.items():
  if len(members) > 1:
    print(f'Cluster {cid} (Grouped): {members}')
  else:
    print(f'Cluster {cid} (Single Feature): {members[0]}')

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import seaborn as sns

# 1. Define groups of correlated features
feature_groups = {
    'Health Status (GenHlth, PhysHlth, DiffWalk)': [
        'GenHlth',
        'PhysHlth',
        'DiffWalk',
    ],
    'Socioeconomic (Income, Education)': ['Income', 'Education'],
    'Cardiovascular (HighBP, HighChol, HeartDisease, Stroke)': [
        'HighBP',
        'HighChol',
        'HeartDiseaseorAttack',
        'Stroke',
    ],
    'Lifestyle (PhysActivity, Fruits, Veggies, Alcohol)': [
        'PhysActivity',
        'Fruits',
        'Veggies',
        'HvyAlcoholConsump',
    ],
}

# 2. Compute Standard Single-Feature PFI
pfi_single = permutation_importance(
    model, X_test, y_test, scoring='roc_auc', n_repeats=10, random_state=42
)
df_single = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': pfi_single.importances_mean,
}).sort_values(by='Importance', ascending=False)

# 3. Compute Grouped PFI (Joint Shuffling)
baseline_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
all_grouped_cols = [col for cols in feature_groups.values() for col in cols]
ungrouped_cols = [col for col in X_test.columns if col not in all_grouped_cols]

grouped_records = []
n_repeats = 10

# Correlated Groups
for group_name, cols in feature_groups.items():
  existing_cols = [c for c in cols if c in X_test.columns]
  if not existing_cols:
    continue
  importances = []
  for r in range(n_repeats):
    X_test_perm = X_test.copy()
    perm_idx = np.random.RandomState(42 + r).permutation(len(X_test))
    for col in existing_cols:
      X_test_perm[col] = X_test[col].values[perm_idx]

    perm_score = roc_auc_score(y_test, model.predict_proba(X_test_perm)[:, 1])
    importances.append(baseline_score - perm_score)

  grouped_records.append({
      'Feature / Group': f'[GROUP] {group_name}',
      'Importance': np.mean(importances),
  })

# Ungrouped Individual Features
for col in ungrouped_cols:
  importances = []
  for r in range(n_repeats):
    X_test_perm = X_test.copy()
    perm_idx = np.random.RandomState(42 + r).permutation(len(X_test))
    X_test_perm[col] = X_test[col].values[perm_idx]

    perm_score = roc_auc_score(y_test, model.predict_proba(X_test_perm)[:, 1])
    importances.append(baseline_score - perm_score)

  grouped_records.append({'Feature / Group': col, 'Importance': np.mean(importances)})

df_grouped = pd.DataFrame(grouped_records).sort_values(
    by='Importance', ascending=False
)

# 4. Plot Side-by-Side Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(
    data=df_single.head(10),
    x='Importance',
    y='Feature',
    hue='Feature',
    ax=ax1,
    palette='Blues_r',
    legend=False,
)
ax1.set_title(
    'Standard Single-Feature PFI\n(Underestimates Correlated Features)',
    fontweight='bold',
)
ax1.set_xlabel('Drop in Test ROC-AUC')

sns.barplot(
    data=df_grouped,
    x='Importance',
    y='Feature / Group',
    hue='Feature / Group',
    ax=ax2,
    palette='Oranges_r',
    legend=False,
)
ax2.set_title(
    'Grouped Permutation Feature Importance\n(Captures Group-Level Impact)',
    fontweight='bold',
)
ax2.set_xlabel('Drop in Test ROC-AUC')

for ax in (ax1, ax2):
  ax.axvline(0, color='gray', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Global audit: drop in held-out ROC AUC after shuffling one feature.
sample = X_test.sample(n=min(2000, len(X_test)), random_state=SEED)
result = permutation_importance(model, sample, y_test.loc[sample.index],
                                scoring='roc_auc', n_repeats=5, random_state=SEED, n_jobs=1)
importance = pd.Series(result.importances_mean, index=features).sort_values()
importance.tail(12).plot.barh(figsize=(8, 5), xlabel='Mean decrease in ROC AUC')
plt.title('Permutation importance on held-out sample'); plt.tight_layout(); plt.show()
display(importance.sort_values(ascending=False).head(15).to_frame('mean_auc_drop'))


## Local explanation: exact decision path

The conditions below show why the tree assigned its leaf score. They do not estimate the effect of changing a feature.

In [ ]:
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt

In [ ]:
# 1. Initialize LimeTabularExplainer on training features
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=features,
    class_names=['No Diabetes', 'Diabetes'],
    mode='classification',
    random_state=42,
)

# 2. Pick a test patient instance (is patient index 23 in the original dataset)
row_position = 0
patient_instance = X_test.iloc[row_position]
patient_df = X_test.iloc[[row_position]]  # 2D DataFrame (preserves feature names)

# 3. Generate LIME explanation (pass .values to suppress LIME pandas warnings)
exp = explainer.explain_instance(
    data_row=patient_instance.values,
    predict_fn=model.predict_proba,
    num_features=5,
)

# 4. Display model prediction & LIME weights
# [0, 1] gets Row 0, Class 1 (Diabetes probability)
prob_diabetes = model.predict_proba(patient_df)[0, 1]

print(
    f'Model predicted probability of Diabetes for Patient {row_position} (23 in original dataset):'
    f' {prob_diabetes:.3f}'
)
print('\nLIME Local Explanation Weights:')
for feat, weight in exp.as_list():
  print(f'  {feat}: {weight:+.4f}')

print(f'\nLIME Local Fidelity (R^2): {exp.score:.4f}')

# 5. Plot LIME horizontal bar plot
fig = exp.as_pyplot_figure()
plt.title(f'LIME Explanation for Decision Tree (Patient {row_position})(23 in original dataset)')
plt.tight_layout()
plt.show()

### change back to python to run

row_position = 0  # Change to inspect another held-out person
row = X_test.iloc[[row_position]]
filled = model[0].transform(row)
tree_model = model[-1]
node_ids = tree_model.decision_path(filled).indices
for node in node_ids:
    feature_index = tree_model.tree_.feature[node]
    if feature_index < 0:
        break
    value = filled[0, feature_index]
    limit = tree_model.tree_.threshold[node]
    relation = '<=' if value <= limit else '>'
    print(f'{features[feature_index]} = {value:g} {relation} {limit:.3g}')
print('Model score:', round(model.predict_proba(row)[0, 1], 3))
print('True survey label:', int(y_test.iloc[row_position]))


## Error audit

Review false negatives and false positives. Features such as `Age`, `Income`, and access to care may encode social patterns; differences in this dataset alone cannot establish fairness or clinical safety.


### change back to python to run

audit = X_test.copy()
audit['actual'] = y_test
audit['score'] = probabilities
audit['predicted'] = (probabilities >= 0.5).astype(int)
display(audit.loc[(audit.actual == 1) & (audit.predicted == 0)].sort_values('score').head(10))
display(audit.loc[(audit.actual == 0) & (audit.predicted == 1)].sort_values('score', ascending=False).head(10))
